# 09 · MILP policy optimization (PuLP + CBC)
Maximize Σ x[i,a]·estimated_incremental_ncm[i,a] subject to: one action per customer · expected promo spend ≤ $62,000 ·
offers ≤ floor(60%·N) · PCT_20 ≤ 15% of offers · a pre-reserved random 5% NO_OFFER measurement holdout (seed 42).
Input: Step 08 customer × action table (+ Step 07 July features for the holdout balance audit). No model training here.

## Load data + constants

In [1]:
import sys, math, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, pulp
from src.common import TABLES, load_processed
from src.modeling import ACTIONS, COMMERCIAL
pd.set_option("display.width", 200)
vals = pd.read_csv(TABLES / "july_customer_action_values.csv", parse_dates=["policy_as_of_ts"])
jf = load_processed("july_features")
BUDGET, MAX_OFFER_SHARE, MAX_PCT20_SHARE, HOLDOUT_SHARE = 62_000, 0.60, 0.15, 0.05
custs = np.sort(vals.customer_id.unique())
N = len(custs)
MAX_OFFERS, N_HOLD = math.floor(MAX_OFFER_SHARE * N), math.ceil(HOLDOUT_SHARE * N)
assert len(vals) == 4 * N
print(f"N={N} | max offers={MAX_OFFERS} | holdout={N_HOLD} | budget=${BUDGET:,}")

N=2776 | max offers=1665 | holdout=139 | budget=$62,000


## Reserve the random NO_OFFER measurement holdout (before optimization, independent of model scores)

In [2]:
hold_ids = np.sort(np.random.default_rng(42).choice(custs, N_HOLD, replace=False))
hold = pd.DataFrame({"customer_id": hold_ids, "measurement_holdout": 1})
hold.to_csv(TABLES / "measurement_holdout.csv", index=False)
bal_cols = ["customer_tenure_days", "is_member", "prior_order_count", "days_since_last_order", "sessions_30d", "days_since_last_session"]
b = jf[["customer_id"] + bal_cols].merge(vals.query("action == 'NO_OFFER'")[["customer_id", "predicted_purchase_probability"]], on="customer_id")
b["holdout"] = b.customer_id.isin(hold_ids)
cols = bal_cols + ["predicted_purchase_probability"]
balance = pd.DataFrame({"holdout_mean": b.loc[b.holdout, cols].mean(), "rest_mean": b.loc[~b.holdout, cols].mean(), "all_mean": b[cols].mean()})
balance["std_mean_diff"] = (balance.holdout_mean - balance.rest_mean) / b[cols].std()
balance.rename_axis("variable").to_csv(TABLES / "measurement_holdout_balance.csv")
balance.round(3)

,holdout_mean,rest_mean,all_mean,std_mean_diff
customer_tenure_days,751.561,728.623,729.772,0.073
is_member,0.245,0.273,0.272,-0.065
prior_order_count,0.173,0.223,0.221,-0.099
days_since_last_order,231.348,250.996,250.303,-0.351
sessions_30d,0.086,0.099,0.099,-0.039
days_since_last_session,146.838,134.829,135.444,0.122
predicted_purchase_probability,0.125,0.131,0.130,-0.085


## Build and solve the MILP

In [3]:
def cbc_solver():
    """Bundled CBC; on Windows-on-ARM with x64 Python, PuLP looks for a missing arm64 binary, so point to the x64 build."""
    exe = Path(pulp.__file__).parent / "solverdir" / "cbc" / "win" / "64" / "cbc.exe"
    return pulp.COIN_CMD(path=str(exe), msg=0) if os.name == "nt" and exe.exists() else pulp.PULP_CBC_CMD(msg=0)

v = vals.set_index(["customer_id", "action"])
x = pulp.LpVariable.dicts("x", [(i, a) for i in custs for a in ACTIONS], cat="Binary")
prob = pulp.LpProblem("july_policy", pulp.LpMaximize)
prob += pulp.lpSum(v.at[(i, a), "estimated_incremental_ncm"] * x[i, a] for i in custs for a in COMMERCIAL)
for i in custs:
    prob += pulp.lpSum(x[i, a] for a in ACTIONS) == 1
for i in hold_ids:
    prob += x[i, "NO_OFFER"] == 1
offers = pulp.lpSum(x[i, a] for i in custs for a in COMMERCIAL)
prob += pulp.lpSum(v.at[(i, a), "estimated_promo_cost"] * x[i, a] for i in custs for a in COMMERCIAL) <= BUDGET
prob += offers <= MAX_OFFERS
prob += pulp.lpSum(x[i, "PCT_20"] for i in custs) <= MAX_PCT20_SHARE * offers
status = pulp.LpStatus[prob.solve(cbc_solver())]
print("solver status:", status, "| objective:", round(pulp.value(prob.objective), 2))
assert status == "Optimal"

solver status: Optimal | objective: 6406.9


## Extract the policy

In [4]:
chosen = [(i, a) for i in custs for a in ACTIONS if x[i, a].value() > 0.5]
pol = pd.DataFrame(chosen, columns=["customer_id", "action"]).merge(vals, on=["customer_id", "action"], how="left")
pol = pol.rename(columns={"action": "recommended_action"}).merge(hold, on="customer_id", how="left").fillna({"measurement_holdout": 0})
pol["measurement_holdout"] = pol.measurement_holdout.astype(int)
pol = pol[["customer_id", "campaign_id", "policy_as_of_ts", "recommended_action", "predicted_ncm", "estimated_incremental_ncm",
           "estimated_promo_cost", "predicted_purchase_probability", "measurement_holdout"]]
pol.to_csv(TABLES / "optimized_policy_detailed.csv", index=False)
pol.recommended_action.value_counts().reindex(ACTIONS, fill_value=0)

recommended_action
NO_OFFER     1111
PCT_10        206
PCT_20          0
FREE_SHIP    1459
Name: count, dtype: int64

## Constraint audit + summary

In [5]:
n_off = int(pol.recommended_action.ne("NO_OFFER").sum()); n20 = int(pol.recommended_action.eq("PCT_20").sum())
spend = pol.estimated_promo_cost.sum()
all_no = vals.query("action == 'NO_OFFER'").predicted_ncm.sum()
audit = pd.DataFrame([
    ("customers", N, N, "=="), ("exactly-one-action violations", int(len(pol) != N or not pol.customer_id.is_unique), 0, "=="),
    ("expected promo spend ($)", round(spend, 2), BUDGET, "<="), ("offered customers", n_off, MAX_OFFERS, "<="),
    ("offered share", round(n_off / N, 4), MAX_OFFER_SHARE, "<="), ("PCT_20 customers", n20, None, "info"),
    ("PCT_20 share among offered", round(n20 / n_off, 4) if n_off else 0.0, MAX_PCT20_SHARE, "<="),
    ("reserved holdout customers", N_HOLD, math.ceil(HOLDOUT_SHARE * N), ">="), ("reserved holdout share", round(N_HOLD / N, 4), HOLDOUT_SHARE, ">="),
    ("holdout customers assigned NO_OFFER", int(pol.query("measurement_holdout == 1").recommended_action.eq("NO_OFFER").sum()), N_HOLD, "=="),
    ("solver status", status, "Optimal", "==")], columns=["check", "value", "limit", "rule"])
ops = {"==": lambda a, b: a == b, "<=": lambda a, b: a <= b + 1e-9, ">=": lambda a, b: a >= b - 1e-9}
audit["pass"] = [True if r == "info" else ops[r](val, lim) for val, lim, r in zip(audit.value, audit.limit, audit.rule)]
audit.to_csv(TABLES / "optimization_constraint_check.csv", index=False)
assert audit["pass"].all(), audit[~audit["pass"]]
summary = pd.DataFrame([{"solver_status": status, "objective_total_expected_incremental_ncm": round(pulp.value(prob.objective), 2),
                         "total_expected_ncm_optimized": round(pol.predicted_ncm.sum(), 2), "total_expected_ncm_all_no_offer": round(all_no, 2),
                         "expected_promo_spend": round(spend, 2), "offered_customers": n_off, "offered_share": round(n_off / N, 4),
                         "pct20_share_among_offered": round(n20 / n_off, 4) if n_off else 0.0, "holdout_customers": N_HOLD,
                         **{f"n_{a}": int((pol.recommended_action == a).sum()) for a in ACTIONS}}])
summary.to_csv(TABLES / "optimization_summary.csv", index=False)
print(audit.to_string(index=False)); summary.T

                              check    value   limit rule  pass
                          customers     2776    2776   ==  True
      exactly-one-action violations        0       0   ==  True
           expected promo spend ($)  1213.65   62000   <=  True
                  offered customers     1665    1665   <=  True
                      offered share   0.5998     0.6   <=  True
                   PCT_20 customers        0    None info  True
         PCT_20 share among offered      0.0    0.15   <=  True
         reserved holdout customers      139     139   >=  True
             reserved holdout share   0.0501    0.05   >=  True
holdout customers assigned NO_OFFER      139     139   ==  True
                      solver status  Optimal Optimal   ==  True


,0
solver_status,Optimal
objective_total_expected_incremental_ncm,6406.9
total_expected_ncm_optimized,51241.42
total_expected_ncm_all_no_offer,44834.52
expected_promo_spend,1213.65
offered_customers,1665
offered_share,0.5998
pct20_share_among_offered,0.0
holdout_customers,139
n_NO_OFFER,1111
